# RSNA Knee — ViT-k LLRD 0.8 (epoch 8) — T4 — TTA x5

Notebook Kaggle **self-contained** (internet off, GPU **T4**) :
- lecture DICOM test → volumes crop 130 mm + bande 20–80 % (24, 224, 224)
- protocole 6 slots (FS/T1, identique v20b). Pas de flip latéralité (train k sans canon L)
- KneeNet **ViT-k** : graphe ViT-e (DINOv3 ViT-B/16, CAM+BiLSTM, stem RGB),
  train `vit_k_llrd08_det_oofmask_full` **epoch 8**
  (gold in-sample 0.940). Poids `vit_k_ep8.pt` (optimizer retiré),
  dataset `sultent/rsna-knee-vit-e-dino-last-weights`
- TTA affine (rotation ±5° + scale 0.95/1.05, hflip off) ×5
- écrit `/kaggle/working/submission.csv` (et le cwd)

Le kernel **doit** écrire `submission.csv`.


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))

    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')

    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(names):
        for name in names:
            for root in search_roots:
                hits = list(root.rglob(name))
                if hits:
                    return hits[0]
        raise FileNotFoundError(f'{names} introuvable (dataset weights attache ?)')

    WEIGHTS_PATH = _find_weight(('vit_k_ep8.pt',))
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_PATH = Path('../kaggle_dataset_vit_e_dino_last/_upload/vit_k_ep8.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('competition dir :', COMP_DIR)
print('poids            :', WEIGHTS_PATH, '| existe :', WEIGHTS_PATH.exists())
print('device           :', DEVICE)


In [ ]:
# -*- coding: utf-8 -*-
"""Pretraitement DICOM -> volume uint8, aligne sur la conversion NPZ.

Recette identique a `kaggle_DICOM_exctract/_gen_convert_nb.py` (grayscale)
et `_gen_convert_rgb25d_nb.py` (voisins DICOM) :
1. Lecture des .dcm, tri InstanceNumber (fallback : normale IOP).
2. RescaleSlope/Intercept, inversion MONOCHROME1.
3. Crop centre 130 mm (PixelSpacing) sinon centre 70 %.
4. Percentiles 1-99 -> [0, 1].
5. Bande profondeur 20-80 % (evite air / bords), puis linspace 24 coupes.
6. Resize IMG_SIZE (224 par defaut, 320 pour crop130_320), uint8.
7. Si rgb_neighbors : pour chaque centre k du linspace, stocker k-1, k, k+1
   dans le volume ORIGINAL (1 coupe DICOM d'ecart), pas dans les 24.
"""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

PROTOCOL_SLOTS: list[tuple[str, int]] = [
    ("Sagittal", 1),
    ("Coronal", 1),
    ("Axial", 1),
    ("Sagittal", 0),
    ("Coronal", 0),
    ("Axial", 0),
]

def laterality_from_study_dir(study_dir):
    raise RuntimeError('laterality off sur ce kernel')

def normalise_laterality(vol, plane, lat):
    raise RuntimeError('laterality off sur ce kernel')


N_SLICES = 24
IMG_SIZE = 224
CROP_MM = 130.0
CROP_REL = 0.70
BAND = (0.20, 0.80)


def _slice_sort_key(ds) -> float:
    """Cle de tri : InstanceNumber, sinon projection sur la normale."""
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> np.ndarray:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray, img_size: int = IMG_SIZE) -> np.ndarray:
    sl = cv2.resize(sl, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def load_series_dicom(
    series_dir: str | Path,
    rgb_neighbors: bool = False,
    img_size: int = IMG_SIZE,
) -> np.ndarray:
    """Un dossier de .dcm -> (24, S, S) ou (24, 3, S, S) uint8, S=`img_size`.

    rgb_neighbors=False : grayscale crop130 (linspace 24).
    rgb_neighbors=True  : R=k-1, G=k, B=k+1 dans le volume original.
    """
    import pydicom

    series_dir = Path(series_dir)
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")

    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f"aucun DICOM lisible dans {series_dir}")
    datasets.sort(key=_slice_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    size = int(img_size)
    if not rgb_neighbors:
        out = np.empty((N_SLICES, size, size), dtype=np.uint8)
        for i, k in enumerate(idx_orig):
            out[i] = _to_u8(vol[int(k)], size)
        return out

    out = np.empty((N_SLICES, 3, size, size), dtype=np.uint8)
    last = n - 1
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        out[i, 0] = _to_u8(vol[km], size)
        out[i, 1] = _to_u8(vol[k], size)
        out[i, 2] = _to_u8(vol[kp], size)
    return out


def _pick_plane_row(series_rows, plane: str):
    """Fluid-sensitive d'abord, sinon n'importe quelle serie du plan."""
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    fluid_col = "Fluid_Sensitive" if "Fluid_Sensitive" in rows.columns else None
    fluid = rows[rows[fluid_col] == 1] if fluid_col else rows.iloc[0:0]
    if len(fluid) == 0 and "Fat_Suppression" in rows.columns:
        fluid = rows[rows.Fat_Suppression == 1]
    pick = (fluid if len(fluid) else rows).sort_values("SeriesInstanceUID").iloc[0]
    return pick


def _pick_nonfluid_row(series_rows, exclude: set[str]):
    """T1 / non-fluide pas deja utilisee. Sagittal d'abord."""
    for plane in ("Sagittal", "Coronal", "Axial"):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if "Fluid_Sensitive" in rows.columns:
            rows = rows[rows.Fluid_Sensitive == 0]
        rows = rows[~rows.SeriesInstanceUID.isin(exclude)]
        if len(rows) == 0:
            continue
        return rows.sort_values("SeriesInstanceUID").iloc[0]
    return None


def _pick_protocol_row(series_rows, plane: str, fluid: int):
    """Un contraste exact (FS ou T1) sur un plan. Pas de repli."""
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if "Fluid_Sensitive" in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and "Fat_Suppression" in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and "Fat_Suppression" in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values("SeriesInstanceUID").iloc[0]


def study_to_tensor(study_dir: str | Path, series_rows,
                    n_series: int = 3,
                    rgb_neighbors: bool = False,
                    img_size: int = IMG_SIZE,
                    laterality: bool = False) -> np.ndarray:
    """Une etude -> float32 dans [0, 1].

    Grayscale : (n_series, 24, S, S) avec S=`img_size`.
    rgb_neighbors : (n_series, 24, 3, S, S) R=k-1 G=k B=k+1 DICOM.

    series_rows : DataFrame des series de l'etude (colonnes SeriesInstanceUID,
    Anatomical_Plane, Fluid_Sensitive, Fat_Suppression).
    n_series=3 : sag / cor / ax, fluid-sensitive d'abord.
    n_series=4 : + T1 extra (sagittal prioritaire).
    n_series=6 : 3 FS + 3 T1 par plan, sans repli (v20b).
    laterality=True : canon gauche (flip R), meme recette que le train.
    """
    planes = ("Sagittal", "Coronal", "Axial")
    study_dir = Path(study_dir)
    size = int(img_size)
    if rgb_neighbors:
        out = np.zeros(
            (n_series, N_SLICES, 3, size, size), dtype=np.float32,
        )
    else:
        out = np.zeros((n_series, N_SLICES, size, size), dtype=np.float32)
    used: set[str] = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i: int, pick, plane: str | None) -> None:
        if pick is None:
            return
        series_dir = study_dir / pick.SeriesInstanceUID
        if not series_dir.exists():
            return
        try:
            vol = load_series_dicom(
                series_dir, rgb_neighbors=rgb_neighbors, img_size=size)
        except Exception as e:
            print(f"  [warn] serie {pick.SeriesInstanceUID}: {e}")
            return
        used.add(str(pick.SeriesInstanceUID))
        vol = vol.astype(np.float32) / 255.0
        if laterality:
            pl = plane or getattr(pick, "Anatomical_Plane", None)
            vol = normalise_laterality(vol, pl, lat)
        out[i] = vol

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        _fill(i, _pick_plane_row(series_rows, plane), plane)
    if n_series >= 4:
        pick = _pick_nonfluid_row(series_rows, used)
        pl = None if pick is None else str(pick.Anatomical_Plane)
        _fill(3, pick, pl)
    return out


In [ ]:
# -*- coding: utf-8 -*-
"""KneeNet-ViT-e — v1 (CAM+BiLSTM) + stem RGB k-1/k/k+1 de d.

Famille CLS (c/d) : le 2.5D aide (contusion 0.66 -> 0.72) mais le gold
reste ~0.74 vs v1 0.80. Le plafond CLS+v8 est sous v1. On recolle le
graphe qui score (CAM patches + LSTM) et on y met le seul levier D
qui a marche.

  - stem : R=k-1, G=k, B=k+1, stem_mix init G (epoch 0 == v1)
  - cartes patch 14x14, CAM + GAP, BiLSTM init 0 (comme v1)
  - LayerScale gele (recette c/d)

Recette : freeze 1, lr_backbone 3e-5 (v1 tenait), re-warmup 1 epoch.
Option `drop_path_rate` (YAML) : stochastic depth timm, OFF pendant
le freeze (epoch 1 = identity e). Defaut 0 = graphe e inchange.
Poids : `weights/dinov3-vitb16-lvd1689m/model.safetensors`
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/dinov3-vitb16-lvd1689m"
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6
PATCH_SIZE = 16
N_PREFIX = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TIMM_NAME = "vit_base_patch16_dinov3.lvd1689m"


def _weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids DINOv3 introuvables : {p.resolve()}")
    return p


class _Dinov3VitEncoder(nn.Module):
    """ViT-B/16 DINOv3 -> cartes (N, 768, H/16, W/16)."""

    def __init__(self, weights_path: str = DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 drop_path_rate: float = 0.0) -> None:
        super().__init__()
        kwargs = dict(
            num_classes=0,
            global_pool="",
            dynamic_img_size=True,
        )
        dp = float(drop_path_rate or 0.0)
        if dp > 0:
            kwargs["drop_path_rate"] = dp
        if pretrained:
            ckpt = _weights_file(weights_path)
            self.backbone = timm.create_model(
                TIMM_NAME,
                pretrained=True,
                pretrained_cfg_overlay={"file": str(ckpt)},
                **kwargs,
            )
            print(f"DINOv3 ViT-B/16 charge : {ckpt}", flush=True)
        else:
            self.backbone = timm.create_model(
                TIMM_NAME, pretrained=False, **kwargs,
            )
        self.backbone.set_grad_checkpointing(True)
        self.num_features = int(self.backbone.num_features)
        self.patch_size = PATCH_SIZE
        self.n_prefix = int(getattr(
            self.backbone, "num_prefix_tokens", N_PREFIX))
        self.drop_path_rate = dp
        self._drop_path: list[tuple[nn.Module, float]] = []
        for m in self.backbone.modules():
            if m.__class__.__name__ == "DropPath":
                self._drop_path.append((m, float(m.drop_prob)))
        if dp > 0:
            print(
                f"drop_path {dp:g} | {len(self._drop_path)} modules "
                f"(off pendant freeze)",
                flush=True,
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (N, 3, H, W) -> (N, C, h, w) patch tokens."""
        tokens = self.backbone.forward_features(x)
        patches = tokens[:, self.n_prefix:]
        n, n_hw, c = patches.shape
        side = int(n_hw ** 0.5)
        if side * side != n_hw:
            raise ValueError(
                f"grille patch non carree : {n_hw} tokens "
                f"(H={x.shape[-2]}, W={x.shape[-1]})"
            )
        return patches.transpose(1, 2).reshape(n, c, side, side)


class KneeNet(nn.Module):
    arch_id = "kneenet_vit_e"

    def __init__(self, backbone: str = TIMM_NAME,
                 n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 4, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN,
                 drop_path_rate: float = 0.0) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.patch_size = PATCH_SIZE
        self.lstm_hidden = int(lstm_hidden)
        self.drop_path_rate = float(drop_path_rate or 0.0)

        self.encoder = _Dinov3VitEncoder(
            weights_path=backbone_weights, pretrained=pretrained,
            drop_path_rate=self.drop_path_rate,
        )
        self._freeze_layer_scale()
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean",
            torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std",
            torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),
            persistent=True,
        )
        dp_txt = (f" | drop_path {self.drop_path_rate:g}"
                  if self.drop_path_rate > 0 else "")
        print(
            f"KneeNet-ViT-e | CAM patch {PATCH_SIZE} | BiLSTM "
            f"| RGB k-1/k/k+1{dp_txt} | dim {feat_dim} | slots {self.n_series}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        """out = in_G -> epoch 0 identique a v1."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    @staticmethod
    def _is_layerscale(name: str) -> bool:
        return name.rsplit(".", 1)[-1].startswith("gamma")

    def _freeze_layer_scale(self) -> None:
        n = 0
        for name, p in self.encoder.named_parameters():
            if self._is_layerscale(name):
                p.requires_grad = False
                n += 1
        print(f"LayerScale gele : {n} tenseurs gamma", flush=True)

    def set_encoder_frozen(self, flag: bool) -> None:
        for name, p in self.encoder.named_parameters():
            p.requires_grad = (not flag) and not self._is_layerscale(name)
        self.stem_mix.weight.requires_grad = not flag
        # DropPath stochastique seulement une fois l'encodeur degele,
        # sinon l'epoch 1 n'est plus identity-e.
        for m, p0 in self.encoder._drop_path:
            m.drop_prob = 0.0 if flag else p0

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            h = rgb.size(-2) // self.patch_size
            w = rgb.size(-1) // self.patch_size
            return rgb.new_zeros(0, c, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc = self._spatial_pool(maps)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        self.eval()
        b, s, d, img_h, img_w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw_h, hw_w = img_h // self.patch_size, img_w // self.patch_size
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw_h, hw_w)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            if spatial.dtype != spat.dtype:
                spatial = spatial.to(dtype=spat.dtype)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw_h, hw_w,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from pathlib import Path

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']

N_TARGETS = len(TARGETS)

# TTA affine : identité + rotation ±5° + scale 0.95/1.05 (hflip off)
TTA_VARIANTS = (
    (0.0, 1.00),
    (5.0, 1.00),
    (-5.0, 1.00),
    (0.0, 0.95),
    (0.0, 1.05),
)


def apply_affine(
    x: torch.Tensor,
    angle_deg: float = 0.0,
    scale: float = 1.0,
    tx: float = 0.0,
    ty: float = 0.0,
) -> torch.Tensor:
    # x : (B, S, D, H, W) dans [0, 1]. Meme affine sur les plans.
    if angle_deg == 0.0 and scale == 1.0 and tx == 0.0 and ty == 0.0:
        return x
    if x.ndim != 5:
        raise ValueError(f"attendu 5D (B,S,D,H,W), recu {tuple(x.shape)}")
    b, s, d, h, w = x.shape
    c = 1

    xf = x.reshape(b * s * d, c, h, w)
    angle = xf.new_tensor(angle_deg * np.pi / 180.0)
    sc = xf.new_tensor(scale)
    cos, sin = torch.cos(angle) / sc, torch.sin(angle) / sc
    theta = xf.new_zeros(b, 2, 3)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin
    theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin
    theta[:, 1, 1] = cos
    theta[:, 1, 2] = ty
    theta = theta.repeat_interleave(s * d, dim=0)
    grid = F.affine_grid(theta, xf.shape, align_corners=False)
    xf = F.grid_sample(xf, grid, align_corners=False, padding_mode="zeros")
    return xf.reshape(b, s, d, h, w)


@torch.no_grad()
def _predict_logits(model, x: torch.Tensor) -> torch.Tensor:
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
        return model(x)


def load_model(weights_path: str | Path) -> tuple[torch.nn.Module, dict]:
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {}) if isinstance(ckpt, dict) else {}

    n_series = int(cfg.get("n_series", 6))
    encode_chunk = int(cfg.get("encode_chunk", 4))
    head_dim = int(cfg.get("head_dim", 512))
    dropout = float(cfg.get("dropout", 0.2))
    max_depth = int(cfg.get("max_depth", 64) or 64)
    lstm_hidden = int(cfg.get("lstm_hidden", 256))
    backbone_weights = str(cfg.get("backbone_weights", "weights/dinov3-vitb16-lvd1689m"))

    # IMPORTANT : pretrained=False pour éviter tout download DINOv3 backbone
    model = KneeNet(
        n_series=n_series,
        n_targets=N_TARGETS,
        head_dim=head_dim,
        dropout=dropout,
        pretrained=False,
        encode_chunk=encode_chunk,
        max_depth=max_depth,
        backbone_weights=backbone_weights,
        lstm_hidden=lstm_hidden,
    )
    missing, unexpected = model.load_state_dict(ckpt["model"], strict=False)
    print('load_state_dict missing :', len(missing), '| unexpected :', len(unexpected))
    gold = ckpt.get('gold_macro_auc')
    gold_txt = 'n/a' if gold is None else round(float(gold), 4)
    print('modele', cfg.get('model'), '| run', cfg.get('run_name'),
          '| epoch', ckpt.get('epoch'), '| gold in-sample', gold_txt,
          '| n_series', n_series)
    model.to(DEVICE).eval()
    return model, cfg

test = pd.read_csv(TEST_CSV)
series = pd.read_csv(TEST_SERIES_CSV)
test_series_dir = Path(TEST_SERIES_DIR)

# prefiltres DataFrame pour accélérer l'extraction des lignes par StudyInstanceUID
series_by_uid = {uid: g for uid, g in series.groupby("StudyInstanceUID", sort=False)}

model, _cfg = load_model(WEIGHTS_PATH)
n_series = int(getattr(model, "n_series", 6))

rows = []
print(f"{len(test)} etudes a predire | TTA x{len(TTA_VARIANTS)}")
for i, uid in enumerate(test.StudyInstanceUID):
    study_dir = test_series_dir / uid
    srows = series_by_uid.get(uid, series.iloc[0:0])

    vol = study_to_tensor(
        study_dir,
        srows,
        n_series=n_series,
        rgb_neighbors=False,
        img_size=224,
    )
    x0 = torch.from_numpy(vol).unsqueeze(0).to(device=DEVICE, dtype=torch.float32)

    probs = []
    for angle_deg, scale in TTA_VARIANTS:
        xv = apply_affine(x0, angle_deg=angle_deg, scale=scale)
        logits = _predict_logits(model, xv)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy()[0])

    prob = np.stack(probs, axis=0).mean(axis=0)
    rows.append([uid] + prob.tolist())
    if (i + 1) % 25 == 0:
        print(f"{i+1}/{len(test)} ok", flush=True)

pred = pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGETS)

# --- submission.csv au format exact de sample_submission --------------------
sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index('StudyInstanceUID').reindex(sample.StudyInstanceUID).reset_index()
sub = sub[list(sample.columns)]
prior = sub[TARGETS].mean()
sub[TARGETS] = sub[TARGETS].fillna(prior)

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
out_csv = out_dir / 'submission.csv'
sub.to_csv(out_csv, index=False)
cwd_csv = Path('submission.csv')
if cwd_csv.resolve() != out_csv.resolve():
    sub.to_csv(cwd_csv, index=False)

print('ecrit', out_csv.resolve(), '| lignes', len(sub), '| existe', out_csv.exists())
print(sub.shape, '| colonnes OK :', list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
assert out_csv.exists() and out_csv.stat().st_size > 0, 'submission.csv manquant'
assert not sub[TARGETS].isna().any().any(), 'NaN dans la submission !'
assert list(sub.columns) == list(sample.columns), 'colonnes != sample_submission'
